# Lab 2.2 – Multi-Tool Research Agent

## Objective

The objective of this lab is to build a Multi-Tool Research Agent using LangChain and Groq.

The agent can analyze user queries, determine which tool is required, execute the selected tool, and return the result.

## Tools Included

- Calculator
- Weather Lookup
- Database Search
- Date Formatter
- Currency Converter
- Text Summarizer

## Technologies

- Python
- LangChain
- Groq
- Pydantic
- Jupyter Notebook

# Introduction

Large Language Models are excellent at understanding natural language but cannot perform external tasks on their own.

By integrating custom tools, an AI system can execute calculations, retrieve information, format data, summarize text, and automate workflows.

This lab demonstrates how multiple tools can work together to solve different user requests.

# Agent Workflow

```

```
User Query

      │
      ▼

Large Language Model

      │
      ▼

Choose Appropriate Tool

      │
      ▼

Execute Tool

      │
      ▼

Receive Tool Result

      │
      ▼
Generate Final Response
```


In [1]:
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq
from langchain_core.tools import tool

load_dotenv()

True

In [2]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM Initialized Successfully!")

Groq LLM Initialized Successfully!


# Creating Tools

The following tools were developed in Day 3 and are reused in this lab.

Instead of rebuilding everything from scratch in a production environment, reusable tools are typically imported from separate modules.

For learning purposes, we will recreate them inside this notebook.

In [3]:
@tool
def calculate(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [4]:
@tool
def weather(city: str) -> str:
    """
    Returns the weather for a given city.
    """

    weather_data = {
        "Islamabad": "Sunny, 42°C",
        "Lahore": "Cloudy, 45°C",
        "Karachi": "Hot, 47°C",
        "Peshawar": "Sunny, 44°C"
    }

    return weather_data.get(city, "Weather data not available.")

In [5]:
@tool
def search_db(topic: str) -> str:
    """
    Searches a simple database.
    """

    database = {
        "python": "Python is a high-level programming language.",
        "langchain": "LangChain is a framework for LLM applications.",
        "groq": "Groq provides fast inference for Large Language Models.",
        "pydantic": "Pydantic validates structured data using Python type hints."
    }

    return database.get(topic.lower(), "No information found.")

In [6]:
from datetime import datetime

@tool
def format_date(date_string: str) -> str:
    """
    Converts YYYY-MM-DD into a readable format.
    """

    try:
        date = datetime.strptime(date_string, "%Y-%m-%d")
        return date.strftime("%d %B %Y")

    except Exception:
        return "Invalid date format."

In [7]:
@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """
    Converts currency using fixed exchange rates.
    """

    rates = {
        ("USD", "PKR"): 285,
        ("PKR", "USD"): 1 / 285,
        ("USD", "EUR"): 0.85,
        ("EUR", "USD"): 1.18,
    }

    key = (from_currency.upper(), to_currency.upper())

    if key not in rates:
        return "Conversion not supported."

    converted = amount * rates[key]

    return f"{converted:.2f} {to_currency.upper()}"

In [8]:
@tool
def summarize_text(text: str) -> str:
    """
    Returns a simple summary by extracting the first sentence.
    """

    sentences = text.split(".")

    if sentences and sentences[0].strip():
        return sentences[0].strip() + "."

    return "No summary available."

# Binding All Tools

LangChain allows multiple tools to be registered with a Large Language Model.

The model decides which tool should be used for a particular user query.

In [9]:
tools = [
    calculate,
    weather,
    search_db,
    format_date,
    convert_currency,
    summarize_text
]

llm_with_tools = llm.bind_tools(tools)

print("All tools registered successfully!")

All tools registered successfully!


# Creating a Manual Tool Execution Loop

The LLM can identify which tool should be used, but it does not execute the tool automatically.

In this lab, we create a simple execution loop that:

1. Sends the user's query to the LLM.
2. Reads the selected tool.
3. Executes the corresponding Python function.
4. Displays the result.

This simulates the behavior of an AI agent.

In [10]:
tool_map = {
    "calculate": calculate,
    "weather": weather,
    "search_db": search_db,
    "format_date": format_date,
    "convert_currency": convert_currency,
    "summarize_text": summarize_text,
}

In [11]:
def execute_query(query):
    response = llm_with_tools.invoke(query)

    if not response.tool_calls:
        print("No tool was selected.")
        print(response.content)
        return

    tool_call = response.tool_calls[0]

    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    print(f"Selected Tool : {tool_name}")
    print(f"Arguments     : {tool_args}")

    tool = tool_map[tool_name]

    result = tool.invoke(tool_args)

    print("\nTool Result:")
    print(result)

# Test 1 – Calculator Tool

The following query should trigger the calculator tool.

In [12]:
execute_query("What is (45 + 15) * 4?")

Selected Tool : calculate
Arguments     : {'expression': '(45 + 15) * 4'}

Tool Result:
240


# Test 2 – Weather Tool

The following query should trigger the weather lookup tool.

In [13]:
execute_query("What is the weather in Lahore?")

Selected Tool : weather
Arguments     : {'city': 'Lahore'}

Tool Result:
Cloudy, 45°C


# Test 3 – Database Search Tool

The following query should trigger the database search tool.

In [14]:
execute_query("Tell me about LangChain.")

Selected Tool : summarize_text
Arguments     : {'text': 'LangChain is an open-source framework that enables developers to build applications powered by large language models like LLaMA, PaLM, and others. It provides a set of tools and APIs to interact with these models, making it easier to integrate them into various projects. LangChain supports a wide range of functionalities, including text generation, conversation management, and data storage. The framework is designed to be flexible and customizable, allowing developers to tailor it to their specific needs. With LangChain, developers can create chatbots, virtual assistants, content generation tools, and more, leveraging the capabilities of large language models.'}

Tool Result:
LangChain is an open-source framework that enables developers to build applications powered by large language models like LLaMA, PaLM, and others.


# Test 4 – Date Formatter Tool

The following query should trigger the date formatter.

In [15]:
execute_query("Format the date 2026-07-02")

Selected Tool : format_date
Arguments     : {'date_string': '2026-07-02'}

Tool Result:
02 July 2026


# Test 5 – Currency Converter

The following query should trigger the currency conversion tool.

In [16]:
execute_query("Convert 100 USD to PKR")

Selected Tool : convert_currency
Arguments     : {'amount': 100, 'from_currency': 'USD', 'to_currency': 'PKR'}

Tool Result:
28500.00 PKR


# Test 6 – Text Summarizer Tool

The following query demonstrates how the agent selects the text summarization tool to produce a concise summary of the given text.

In [17]:
execute_query("""
Summarize this text:

Artificial Intelligence is transforming industries across the world by enabling automation,
improving decision-making, and increasing efficiency. It is widely used in healthcare,
finance, education, and manufacturing.
""")

Selected Tool : summarize_text
Arguments     : {'text': 'Artificial Intelligence is transforming industries across the world by enabling automation, improving decision-making, and increasing efficiency. It is widely used in healthcare, finance, education, and manufacturing.'}

Tool Result:
Artificial Intelligence is transforming industries across the world by enabling automation, improving decision-making, and increasing efficiency.


# Testing Multiple Queries

The agent can process different user requests by selecting the appropriate tool for each query.

The following examples demonstrate how a single agent can handle multiple tasks.

In [18]:
queries = [
    "Calculate 125 * 8",
    "What is the weather in Islamabad?",
    "Tell me about Python.",
    "Format the date 2026-12-25",
    "Convert 200 USD to PKR"
]

for i, query in enumerate(queries, start=1):
    print("=" * 80)
    print(f"Query {i}: {query}")
    execute_query(query)
    print()

Query 1: Calculate 125 * 8
Selected Tool : calculate
Arguments     : {'expression': '125 * 8'}

Tool Result:
1000

Query 2: What is the weather in Islamabad?
Selected Tool : weather
Arguments     : {'city': 'Islamabad'}

Tool Result:
Sunny, 42°C

Query 3: Tell me about Python.
Selected Tool : search_db
Arguments     : {'topic': 'Python'}

Tool Result:
Python is a high-level programming language.

Query 4: Format the date 2026-12-25
Selected Tool : format_date
Arguments     : {'date_string': '2026-12-25'}

Tool Result:
25 December 2026

Query 5: Convert 200 USD to PKR
Selected Tool : convert_currency
Arguments     : {'amount': 200, 'from_currency': 'USD', 'to_currency': 'PKR'}

Tool Result:
57000.00 PKR



# Advantages of Multi-Tool Agents

Multi-tool agents extend the capabilities of Large Language Models by allowing them to interact with external functions.

Some advantages include:

- Better accuracy for calculations
- Ability to retrieve external information
- Modular and reusable tool design
- Improved automation
- Easier integration with APIs and databases
- More reliable responses for specialized tasks

# Limitations

Although tool-calling agents are powerful, they also have some limitations:

- Tool descriptions must be clear for accurate selection.
- Incorrect tool design may produce unexpected results.
- Some tools require external APIs or internet connectivity.
- Complex workflows may require multiple sequential tool calls.
- Proper error handling is essential for production systems.

# Observations

During this lab, the following observations were made:

- The LLM successfully identified the appropriate tool for different user queries.
- Custom Python functions were effectively converted into LangChain tools.
- The manual execution loop simulated the behaviour of a simple AI agent.
- Different tools handled different categories of tasks independently.
- The modular design makes it easy to add new tools in the future.

# Conclusion

In this lab, a Multi-Tool Research Agent was developed using LangChain and Groq.

The agent was capable of selecting and executing different tools based on user requests, including calculations, weather lookup, database search, date formatting, currency conversion, and text summarization.

This lab demonstrates the importance of tool calling in modern AI systems and provides a strong foundation for building autonomous AI agents capable of interacting with external services and performing complex workflows.